# 00 — Getting started: build your first weather file

**Learning objectives:** install and verify `smhi2epw`, assemble a minimal `EPWConfig`, compile one Gothenburg AMY, and interpret the returned quality report.

**Prerequisites:** Python 3.11+, `pip install -e ".[tutorials]"`, and internet access to SMHI and, if recovery needs it, Open-Meteo. **Expected runtime:** 2–8 minutes on the first run; cached reruns are faster. The notebook writes only to `examples/output/`.

## Installation check

Use a source checkout with `python -m pip install -e ".[tutorials]"`, then launch `jupyter lab examples/`. No PyPI release was available when checked on 6 October 2026. If the import below fails, confirm that this notebook's kernel uses the same virtual environment as `pip`.


In [ ]:
import sys
from pathlib import Path

import smhi2epw
from smhi2epw import EPWConfig, compile_epw

print("Python:", sys.version.split()[0])
print("smhi2epw:", smhi2epw.__version__)
OUTPUT_DIR = Path("examples/output") if Path("examples").exists() else Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.resolve()

## Assemble the request

Station 71420 supplies observed temperature, humidity, pressure, and wind. The explicit city-centre coordinates control STRÅNG, solar geometry, and the EPW location header. `utc_offset=1` means Swedish Local Standard Time; daylight saving is intentionally not used in EPW calendars.

In [ ]:
config = EPWConfig(
    year=2023,
    output_path=str(OUTPUT_DIR / "gothenburg_2023.epw"),
    station_id=71420,
    city="Gothenburg",
    latitude=57.7156,
    longitude=11.9924,
    utc_offset=1,
)
result = compile_epw(config)
result

In [ ]:
summary = {
    "rows": result.rows,
    "station": (
        f"{result.station.station_id} — {result.station.name}"
        if result.station
        else "Station metadata unavailable"
    ),
    "station_distance_km": (
        round(result.coordinate_distance_km, 2)
        if result.coordinate_distance_km is not None
        else None
    ),
    "solar_source": result.report.solar_source,
    "weather_classification": result.report.weather_classification,
    "required_reconstructed_fraction": result.report.required_reconstructed_fraction,
    "source_fractions": result.report.source_fractions,
    "reanalysis_filled_hours": result.report.reanalysis_filled_hours,
    "warnings": result.report.warnings,
    "interpolated_percent": round(100 * result.interpolated_fraction, 3),
    "max_solar_residual_W_m2": result.report.energy_balance_max_residual,
}
summary

## Key takeaways and exercises

A non-leap EPW has 8760 rows. Inspect `result.report.max_gap_hours`, `linear_filled_hours`, and `diurnal_filled_hours` before using the file for calibration. Interpolation excludes donor and ERA5 recovery. Inspect `source_fractions`, `required_reconstructed_fraction`, and warnings, and retain the adjacent `.epw.json` receipt. Complete weather does not guarantee accurate local event peaks.

**Common mistakes:** using a daylight-saving offset, confusing the SMHI ID with a WMO ID, deleting the cache before every run, or assuming station coordinates describe the building.

**Try it:** change only the target year; then omit `station_id` and let the coordinates select a station, preferring full-year coverage when available. Predict which fields remain tied to the station and which follow the coordinates.